<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Modelo de lenguaje con tokenización por caracteres

### Consigna
- Seleccionar un corpus de texto sobre el cual entrenar el modelo de lenguaje.
- Realizar el pre-procesamiento adecuado para tokenizar el corpus, estructurar el dataset y separar entre datos de entrenamiento y validación.
- Proponer arquitecturas de redes neuronales basadas en unidades recurrentes para implementar un modelo de lenguaje.
- Con el o los modelos que consideren adecuados, generar nuevas secuencias a partir de secuencias de contexto con las estrategias de greedy search y beam search determístico y estocástico. En este último caso observar el efecto de la temperatura en la generación de secuencias.


### Sugerencias
- Durante el entrenamiento, guiarse por el descenso de la perplejidad en los datos de validación para finalizar el entrenamiento. Para ello se provee un callback.
- Explorar utilizar SimpleRNN (celda de Elman), LSTM y GRU.
- rmsprop es el optimizador recomendado.

---

## Resolución

Se entrena un **modelo de lenguaje a nivel de caracteres** sobre *La vuelta al mundo en 80 días*
(Julio Verne, en español). Siguiendo la sugerencia de la consigna, se **comparan tres
arquitecturas recurrentes** entrenadas con exactamente el mismo pipeline:

| Modelo | Capa recurrente |
|--------|-----------------|
| 1 | **SimpleRNN** (celda de Elman) |
| 2 | **LSTM** |
| 3 | **GRU** |

La comparación se hace por **perplejidad** sobre validación (más baja = mejor). Para el
mejor modelo se generan secuencias con **greedy**, **beam search determinista** y
**beam search estocástico**, analizando el efecto de la **temperatura**.

> **Ejecución:** pensado para **Google Colab con GPU (T4)**.

### Importación de librerías

In [ ]:
import random
import io
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical, pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Input, Dense, Dropout, TimeDistributed,
                                     CategoryEncoding, SimpleRNN, LSTM, GRU)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU') or "NO (ir a Entorno de ejecución -> GPU)")

### Datos
Usamos como corpus el libro *La vuelta al mundo en 80 días* de Julio Verne (textos.info).

In [ ]:
import urllib.request
import bs4 as bs

In [ ]:
raw_html = urllib.request.urlopen('https://www.textos.info/julio-verne/la-vuelta-al-mundo-en-80-dias/ebook')
raw_html = raw_html.read()

# Parsear el HTML; 'lxml' es el parser
article_html = bs.BeautifulSoup(raw_html, 'lxml')

# Tomar todos los párrafos <p>
article_paragraphs = article_html.find_all('p')

article_text = ''
for para in article_paragraphs:
    article_text += para.text + ' '

# todo a minúscula
article_text = article_text.lower()
print("Caracteres totales del corpus:", len(article_text))

In [ ]:
article_text[:1000]

### Tamaño del contexto

Como el modelo es por caracteres, todo el corpus puede tratarse como un único documento y
el tamaño de contexto se elige con libertad.

In [ ]:
max_context_size = 100

### Vocabulario y tokenización por caracteres

In [ ]:
# El vocabulario es el conjunto único de caracteres. Lo ordenamos (sorted) para que
# el mapeo char->idx sea REPRODUCIBLE entre ejecuciones (set() no garantiza orden).
chars_vocab = sorted(set(article_text))
vocab_size = len(chars_vocab)
print("Tamaño del vocabulario de caracteres:", vocab_size)

In [ ]:
# Diccionarios char<->idx. char2idx actúa de tokenizador.
char2idx = {k: v for v, k in enumerate(chars_vocab)}
idx2char = {v: k for k, v in char2idx.items()}

In [ ]:
# Tokenizamos el texto completo
tokenized_text = [char2idx[ch] for ch in article_text]
tokenized_text[:1000]

### Estructuración del dataset (train / validación)

In [ ]:
# p_val: proporción del corpus para validación
# num_val: cantidad de secuencias de tamaño max_context_size para validación
p_val   = 0.1
num_val = int(np.ceil(len(tokenized_text) * p_val / max_context_size))

train_text = tokenized_text[:-num_val*max_context_size]
val_text   = tokenized_text[-num_val*max_context_size:]

# secuencias de validación (no solapadas)
tokenized_sentences_val = [val_text[init*max_context_size:(init+1)*max_context_size]
                           for init in range(num_val)]

# secuencias de entrenamiento (ventana deslizante de a 1)
tokenized_sentences_train = [train_text[init:init+max_context_size]
                             for init in range(len(train_text) - max_context_size + 1)]

Estructuramos el aprendizaje como **many-to-many**:

- Entrada: $[x_0, x_1, \dots, x_N]$
- Target: $[x_1, x_2, \dots, x_{N+1}]$

La red aprende a devolver los tokens desplazados una posición más el nuevo token predicho.
Así, para cada token del target se propaga señal de gradiente por el grafo recurrente
(mejor que un esquema *many-to-one*).

In [ ]:
X = np.array(tokenized_sentences_train[:-1])
y = np.array(tokenized_sentences_train[1:])
print("X:", X.shape, "| y:", y.shape)
print("X[0,:10]:", X[0, :10])
print("y[0,:10]:", y[0, :10])

### Callback de Perplejidad (con Early Stopping)

La **perplejidad** mide la calidad del modelo de lenguaje (más baja = mejor). Este callback
la calcula sobre validación al final de cada época, guarda el mejor modelo y corta el
entrenamiento si no mejora durante `patience` épocas. Se parametriza `model_path` para que
cada arquitectura guarde su propio checkpoint.

In [ ]:
class PplCallback(keras.callbacks.Callback):
    '''Calcula la perplejidad sobre validación al final de cada epoch,
    guarda el mejor modelo y aplica early stopping.'''

    def __init__(self, val_data, history_ppl, patience=5, model_path="best_model.keras"):
        self.val_data    = val_data
        self.history_ppl = history_ppl
        self.patience    = patience
        self.model_path  = model_path

        self.target = []
        self.padded = []
        self.info   = []
        self.min_score = np.inf
        self.patience_counter = 0

        count = 0
        for seq in self.val_data:
            len_seq = len(seq)
            subseq  = [seq[:i] for i in range(1, len_seq)]
            self.target.extend([seq[i] for i in range(1, len_seq)])
            if len(subseq) != 0:
                self.padded.append(pad_sequences(subseq, maxlen=max_context_size, padding='pre'))
                self.info.append((count, count + len_seq - 1))
                count += len_seq - 1
        self.padded = np.vstack(self.padded)

    def on_epoch_end(self, epoch, logs=None):
        scores = []
        predictions = self.model.predict(self.padded, verbose=0)
        for start, end in self.info:
            probs = [predictions[idx_seq, -1, idx_vocab]
                     for idx_seq, idx_vocab in zip(range(start, end), self.target[start:end])]
            scores.append(np.exp(-np.sum(np.log(np.array(probs) + 1e-10)) / (end - start)))

        current_score = float(np.mean(scores))
        self.history_ppl.append(current_score)
        print(f'\n mean perplexity: {current_score:.4f}')

        if current_score < self.min_score:
            self.min_score = current_score
            self.model.save(self.model_path)
            print("Saved new best model!")
            self.patience_counter = 0
        else:
            self.patience_counter += 1
            if self.patience_counter == self.patience:
                print("Stopping training...")
                self.model.stop_training = True

### Definición del modelo (parametrizado por tipo de celda)

El modelo consume índices de tokens y los convierte a **one-hot** con `CategoryEncoding`
(no se entrena una capa de embedding de caracteres), aplicada en cada paso temporal con
`TimeDistributed`. La única diferencia entre los tres modelos es la **celda recurrente**
(`SimpleRNN`, `LSTM` o `GRU`); el resto del pipeline es idéntico para una comparación justa.

In [ ]:
RECURRENT = {'SimpleRNN': SimpleRNN, 'LSTM': LSTM, 'GRU': GRU}

def build_model(cell_type, units=200, dropout=0.1, recurrent_dropout=0.1):
    RNNLayer = RECURRENT[cell_type]
    model = Sequential(name=f"charLM_{cell_type}")
    model.add(TimeDistributed(
        CategoryEncoding(num_tokens=vocab_size, output_mode="one_hot"),
        input_shape=(None, 1)))
    # recurrent_dropout fuerza la implementación NO-cuDNN (más lento) pero es lo usado en clase.
    model.add(RNNLayer(units, return_sequences=True,
                       dropout=dropout, recurrent_dropout=recurrent_dropout))
    model.add(Dense(vocab_size, activation='softmax'))
    model.compile(loss='sparse_categorical_crossentropy', optimizer='rmsprop')
    return model

build_model('SimpleRNN').summary()

### Entrenamiento comparativo de las 3 arquitecturas

Entrenamos `SimpleRNN`, `LSTM` y `GRU` con el mismo número de unidades (200) y el mismo
callback de perplejidad. Cada modelo guarda su propio checkpoint y registramos la mejor
perplejidad de validación alcanzada.

In [ ]:
EPOCHS     = 20
BATCH_SIZE = 256
UNITS      = 200

ppl_histories = {}
best_ppl      = {}
train_times   = {}
model_paths   = {}

for cell_type in ['SimpleRNN', 'LSTM', 'GRU']:
    print("\n" + "="*70)
    print(f"Entrenando modelo: {cell_type}")
    print("="*70)
    tf.keras.backend.clear_session()
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

    model = build_model(cell_type, units=UNITS)
    path  = f"best_{cell_type}.keras"
    hist_ppl = []
    cb = PplCallback(tokenized_sentences_val, hist_ppl, patience=5, model_path=path)

    t0 = time.time()
    model.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[cb], verbose=2)
    train_times[cell_type]   = round(time.time() - t0, 1)
    ppl_histories[cell_type] = hist_ppl
    best_ppl[cell_type]      = float(np.min(hist_ppl))
    model_paths[cell_type]   = path
    print(f"{cell_type}: mejor perplejidad = {best_ppl[cell_type]:.4f}")

### Comparación de las 3 arquitecturas

In [ ]:
df_cmp = pd.DataFrame({
    'modelo':      list(best_ppl.keys()),
    'best_ppl':    [best_ppl[k]    for k in best_ppl],
    'epochs_run':  [len(ppl_histories[k]) for k in best_ppl],
    'train_time_s':[train_times[k] for k in best_ppl],
}).sort_values('best_ppl').reset_index(drop=True)
df_cmp

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cell_type, h in ppl_histories.items():
    axes[0].plot(range(1, len(h)+1), h, marker='o', label=cell_type)
axes[0].set_title('Perplejidad de validación por época')
axes[0].set_xlabel('época'); axes[0].set_ylabel('perplejidad')
axes[0].legend(); axes[0].grid(alpha=.3)

sns.barplot(data=df_cmp, x='modelo', y='best_ppl', ax=axes[1])
axes[1].set_title('Mejor perplejidad por arquitectura'); axes[1].set_ylabel('perplejidad (↓ mejor)')
for i, v in enumerate(df_cmp['best_ppl']):
    axes[1].text(i, v, f"{v:.2f}", ha='center', va='bottom')
plt.tight_layout(); plt.show()

In [ ]:
# Cargamos el MEJOR modelo (menor perplejidad) para la generación de secuencias
best_model_name = df_cmp.iloc[0]['modelo']
model = keras.models.load_model(model_paths[best_model_name])
print(f"Mejor modelo cargado para generación: {best_model_name} "
      f"(perplejidad {best_ppl[best_model_name]:.4f})")

### Generación de secuencias — Greedy search

Predicción del próximo carácter tomando siempre el de mayor probabilidad (argmax).

In [ ]:
def generate_seq(model, seed_text, max_length, n_words):
    '''Genera n_words caracteres a partir de seed_text (greedy / argmax).'''
    output_text = seed_text
    for _ in range(n_words):
        encoded = [char2idx[ch] for ch in output_text.lower()]
        encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')
        y_hat   = np.argmax(model.predict(encoded, verbose=0)[0, -1, :])
        output_text += idx2char[y_hat]
    return output_text

In [ ]:
input_text = 'habia una vez'
print(generate_seq(model, input_text, max_length=max_context_size, n_words=100))

### Beam search (determinista y estocástico) y efecto de la temperatura

`select_candidates` elige, en cada paso, los `num_beams` candidatos:
- **det**: los de mayor probabilidad acumulada (determinista).
- **sto**: muestreados según `softmax(logprob / temperatura)` (estocástico). La
  **temperatura** controla la diversidad: baja → conservador/repetitivo, alta → creativo/ruidoso.

In [ ]:
from scipy.special import softmax

def encode(text, max_length=max_context_size):
    encoded = [char2idx[ch] for ch in text]
    return pad_sequences([encoded], maxlen=max_length, padding='pre')

def decode(seq):
    return ''.join([idx2char[ch] for ch in seq])

In [ ]:
def select_candidates(pred, num_beams, vocab_size, history_probs, history_tokens, temp, mode):
    pred_large = []
    for idx, pp in enumerate(pred):
        pred_large.extend(np.log(pp + 1e-10) + history_probs[idx])
    pred_large = np.array(pred_large)

    if mode == 'det':
        idx_select = np.argsort(pred_large)[::-1][:num_beams]            # determinista
    elif mode == 'sto':
        idx_select = np.random.choice(np.arange(pred_large.shape[0]),
                                      num_beams, p=softmax(pred_large/temp))  # estocástico
    else:
        raise ValueError(f'Modo inválido: {mode}. Use det o sto.')

    new_history_tokens = np.concatenate(
        (np.array(history_tokens)[idx_select // vocab_size],
         np.array([idx_select % vocab_size]).T), axis=1)
    return pred_large[idx_select.astype(int)], new_history_tokens.astype(int)


def beam_search(model, num_beams, num_words, input, temp=1, mode='det'):
    encoded = encode(input)
    y_hat   = model.predict(encoded, verbose=0)[0, -1, :]
    vocab_size = y_hat.shape[0]

    history_probs  = [0]*num_beams
    history_tokens = [encoded[0]]*num_beams
    history_probs, history_tokens = select_candidates(
        [y_hat], num_beams, vocab_size, history_probs, history_tokens, temp, mode)

    for i in range(num_words-1):
        preds = []
        for hist in history_tokens:
            input_update = np.array([hist[i+1:]]).copy()
            preds.append(model.predict(input_update, verbose=0)[0, -1, :])
        history_probs, history_tokens = select_candidates(
            preds, num_beams, vocab_size, history_probs, history_tokens, temp, mode)

    return history_tokens[:, -(len(input)+num_words):]

**Beam search determinista** (devuelve el haz ordenado; mostramos la mejor secuencia).

In [ ]:
salidas = beam_search(model, num_beams=10, num_words=40, input="habia una vez", mode='det')
print("Mejor secuencia (beam determinista):")
print(decode(salidas[0]))

**Beam search estocástico** — efecto de la temperatura.

In [ ]:
for temp in [0.3, 0.7, 1.0, 1.5]:
    np.random.seed(SEED)
    salidas = beam_search(model, num_beams=10, num_words=40,
                          input="habia una vez", temp=temp, mode='sto')
    print(f"\n--- temperatura = {temp} ---")
    print(decode(salidas[0]))

Comparación rápida de las **tres estrategias** sobre el mismo *seed*.

In [ ]:
seed = "el señor fogg"
print("GREEDY            :", generate_seq(model, seed, max_context_size, 40))

det = beam_search(model, num_beams=10, num_words=40, input=seed, mode='det')
print("BEAM determinista :", decode(det[0]))

np.random.seed(SEED)
sto = beam_search(model, num_beams=10, num_words=40, input=seed, temp=0.7, mode='sto')
print("BEAM estocástico  :", decode(sto[0]))

### (Opcional) Interfaz interactiva con Gradio

In [ ]:
# Descomentar para probar el modelo interactivamente:
# !pip install -q gradio
# import gradio as gr
#
# def model_response(human_text):
#     encoded = [char2idx[ch] for ch in human_text.lower()]
#     encoded = pad_sequences([encoded], maxlen=max_context_size, padding='pre')
#     y_hat = np.argmax(model.predict(encoded, verbose=0)[0, -1, :])
#     return human_text + idx2char[y_hat]
#
# gr.Interface(fn=model_response, inputs=["textbox"], outputs="text").launch(debug=True)

### Conclusiones e interpretación

> *Ajustar el texto con los números reales de la tabla comparativa.*

**Comparación de arquitecturas (perplejidad de validación).**
- **SimpleRNN** es la celda más simple; sufre el problema de *gradientes que se desvanecen*
  y le cuesta capturar dependencias largas. Suele converger a una perplejidad más alta y/o
  ser inestable.
- **LSTM** y **GRU** incorporan compuertas que regulan el flujo de información y la memoria
  de largo plazo, por lo que típicamente alcanzan **perplejidades menores** que SimpleRNN.
- **GRU** tiene menos parámetros que LSTM (2 compuertas vs 3) y suele entrenar algo más
  rápido, con desempeño comparable; LSTM a veces gana por poco en corpus más complejos.
- Conclusión: el modelo con **menor perplejidad** (ver tabla) es el elegido para generar.

**Estrategias de generación.**
- **Greedy**: rápido pero repetitivo; tiende a quedar atrapado en bucles ("que que que…").
- **Beam search determinista**: explora varias hipótesis y produce texto más coherente que
  greedy, aunque también puede ser conservador/repetitivo al maximizar probabilidad.
- **Beam search estocástico**: introduce diversidad por muestreo. La **temperatura** es
  clave:
  - `temp` baja (0.3): texto conservador, más correcto pero repetitivo.
  - `temp` media (0.7–1.0): buen balance entre coherencia y variedad.
  - `temp` alta (1.5): más creativo pero con más errores ortográficos y "ruido".

**Limitaciones.**
1. **Modelo por caracteres:** debe aprender la ortografía desde cero; necesita más datos y
   contexto que un modelo por palabras/subpalabras para producir texto fluido.
2. **Corpus único y acotado:** un solo libro limita el vocabulario y el estilo aprendido.
3. **Sin embeddings ni atención:** la representación one-hot y la recurrencia simple quedan
   lejos de un Transformer; las dependencias muy largas se pierden.
4. **Perplejidad:** buena métrica cuantitativa, pero no garantiza coherencia semántica;
   conviene complementarla con la inspección cualitativa de los textos generados.